### Test Reward Model

In [3]:

import torch
import random
import numpy as np
from speculative_functions import *
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

def set_seed(seed: int = 42):
    """Locks all random seeds for complete reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    
    # If you are using a GPU, you also need to lock the CUDA seeds
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        # Forces cuDNN to use deterministic algorithms
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

# Call this before doing anything else!
set_seed(42)

In [4]:
import pandas as pd

splits = {'train': 'main/train-00000-of-00001.parquet', 'test': 'main/test-00000-of-00001.parquet'}
df = pd.read_parquet("hf://datasets/openai/gsm8k/" + splits["train"])

In [5]:
df.head()

,question,answer
0,Natalia sold clips to 48 of her friends in Apr...,Natalia sold 48/2 = <<48/2=24>>24 clips in May...
1,Weng earns $12 an hour for babysitting. Yester...,Weng earns 12/60 = $<<12/60=0.2>>0.2 per minut...
2,Betty is saving money for a new wallet which c...,"In the beginning, Betty has only 100 / 2 = $<<..."
3,"Julie is reading a 120-page book. Yesterday, s...",Maila read 12 x 2 = <<12*2=24>>24 pages today....
4,James writes a 3-page letter to 2 different fr...,He writes each friend 3*2=<<3*2=6>>6 pages a w...


In [6]:
df['question'] = df['question'].str.replace('\n', ' ')

In [7]:
df['question'].iloc[0]

'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?'

In [8]:
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

In [9]:
target_id = "HuggingFaceTB/SmolLM2-360M-Instruct"
draft_id = "HuggingFaceTB/SmolLM2-135M-Instruct"

In [10]:
tokenizer = AutoTokenizer.from_pretrained(target_id)
target_model = AutoModelForCausalLM.from_pretrained(target_id, torch_dtype=dtype).to(device)
draft_model = AutoModelForCausalLM.from_pretrained(draft_id, torch_dtype=dtype).to(device)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

In [ ]:
messages = [{"role": "user", "content": df['question'].iloc[1]}]
# We'll name this 'inputs' to make the distinction clear
inputs = tokenizer.apply_chat_template(
    messages, 
    add_generation_prompt=True, 
    return_tensors="pt",
    return_dict=True # Good practice to be explicit when extracting dict keys
).to(device)

# Extract the actual tensor from the BatchEncoding dictionary
input_ids = inputs["input_ids"]

# Now input_ids is a standard tensor, and [0] gets the first sequence
decoded_text = tokenizer.decode(input_ids[0])
print(decoded_text)

<|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>user
Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did she earn?<|im_end|>
<|im_start|>assistant



In [16]:
# Generate tokens from the model
outputs = target_model.generate(
    **inputs, 
    max_new_tokens=256,
    do_sample=True,
    temperature=0.7
)

# Extract ONLY the newly generated tokens (skipping the prompt)
prompt_length = inputs["input_ids"].shape[-1]
new_tokens = outputs[0][prompt_length:]

# Decode the response to text
response = tokenizer.decode(new_tokens, skip_special_tokens=True)
print(response)

To find out how much Weng earned yesterday, we need to know how many hours she worked. Since she just did 50 minutes of babysitting, we can convert the amount of time to hours.

50 minutes / 60 minutes per hour = 0.83 hours

Now that we know Weng worked for 0.83 hours, we can find out how much she earned by multiplying the hours worked by the hourly rate.

0.83 hours * $12 per hour = $10.59

So, Weng earned $10.59 yesterday.


In [17]:
df['answer'].iloc[1]

'Weng earns 12/60 = $<<12/60=0.2>>0.2 per minute.\nWorking 50 minutes, she earned 0.2 x 50 = $<<0.2*50=10>>10.\n#### 10'

In [1]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import os
from huggingface_hub import login

login(os.environ.get("HF_TOKEN"))


reward_model_name = "Skywork/Skywork-Reward-Llama-3.1-8B"

tokenizer = AutoTokenizer.from_pretrained(reward_model_name)
model = AutoModelForSequenceClassification.from_pretrained(reward_model_name)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [4]:
# 2. Define the prompt and the two responses you want to compare
prompt = "Explain why the sky is blue in one simple sentence."
response_good = "The sky is blue because gases in the Earth's atmosphere scatter sunlight in all directions, and blue light is scattered more than other colors because it travels as shorter, smaller waves."
response_bad = "I think the sky is blue because the ocean reflects its color onto the clouds, but I'm not really sure."

# 3. Format inputs into the chat templates required by the model
conv_good = [
    {"role": "user", "content": prompt},
    {"role": "assistant", "content": response_good}
]

conv_bad = [
    {"role": "user", "content": prompt},
    {"role": "assistant", "content": response_bad}
]

# 4. Tokenize (Notice return_dict=True to create the BatchEncoding properly)
inputs_good = tokenizer.apply_chat_template(
    conv_good, 
    tokenize=True, 
    return_tensors="pt",
    return_dict=True
).to(model.device)

inputs_bad = tokenizer.apply_chat_template(
    conv_bad, 
    tokenize=True, 
    return_tensors="pt",
    return_dict=True
).to(model.device)

# 5. Run inference
model.eval()
with torch.no_grad():
    # Notice the ** to unpack the dictionary into input_ids and attention_mask
    score_good = model(**inputs_good).logits[0][0].item()
    score_bad = model(**inputs_bad).logits[0][0].item()

# 6. Compare the results
print(f"Good response score: {score_good:.4f}")
print(f"Bad response score:  {score_bad:.4f}")

if score_good > score_bad:
    print("\nResult: The model prefers the GOOD response.")
else:
    print("\nResult: The model prefers the BAD response.")

Good response score: -6.2812
Bad response score:  -31.3750

Result: The model prefers the GOOD response.
